If only graph data is available, we train it using this script

In [ ]:
import os

import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from data_setup import TimTremorGraph, GRAPH_TRANSFORM
from model_builder import *
import csv
from datetime import datetime
import json
import pandas as pd

# try:
#     from torchinfo import summary
# except:
#     print("[INFO] Couldn't find torchinfo... installing it.")
#     !pip install -q torchinfo
from torchinfo import summary

# try:
#     from torchmetrics import Accuracy, Precision, Recall, F1Score, ConfusionMatrix
# except:
#     print("[INFO] Couldn't find torchmetrics... installing it.")
#     !pip install -q torchmetrics
from torchmetrics import Accuracy, Precision, Recall, F1Score, ConfusionMatrix

# try:
#     from tqdm.auto import tqdm
# except:
#     print("[INFO] Couldn't find tqdm... installing it.")
#     !pip install -q tqdm
from tqdm.auto import tqdm

from timeit import default_timer as timer 

In [ ]:
HYPER_PARAMS = {
    'K': 3,
    'LR': 1e-3,
    'BATCH_SIZE': 16,
    'EPOCHS': 1
}

In [ ]:
class PerformanceMetric:
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.precision = Precision(task="multiclass", num_classes=num_classes, average='macro')
        self.recall = Recall(task="multiclass", num_classes=num_classes, average='macro')
        self.f1 = F1Score(task="multiclass", num_classes=num_classes, average='macro')
        self.confusion_matrix = ConfusionMatrix(task="multiclass", num_classes=num_classes)

    def update(self, predictions, targets):
        self.accuracy.update(predictions, targets)
        self.precision.update(predictions, targets)
        self.recall.update(predictions, targets)
        self.f1.update(predictions, targets)
        self.confusion_matrix.update(predictions, targets)

    def compute(self):
        accuracy = self.accuracy.compute().item()
        precision = self.precision.compute().item()
        recall = self.recall.compute().item()
        f1 = self.f1.compute().item()
        confusion_matrix = self.confusion_matrix.compute().cpu()

        return accuracy, precision, recall, f1, confusion_matrix

    def reset(self):
        self.accuracy.reset()
        self.precision.reset()
        self.recall.reset()
        self.f1.reset()
        self.confusion_matrix.reset()

In [ ]:
def collate_fn(batch):
    landmarks = []
    graph_types = []
    labels = []

    for item in batch:
        landmark, graph_type, label = item
        landmarks.append(landmark)
        graph_types.append(graph_type)
        labels.append(label)

    # Stack landmark tensors into a single tensor
    landmarks = torch.stack(landmarks)

    # Convert graph types and labels to tensors
    graph_types = torch.tensor(graph_types)
    labels = torch.tensor(labels)

    return landmarks, graph_types, labels


In [ ]:
class Trainer:
    def __init__(self,
                 data_root_dir:str,
                 train_instance,
                 val_instance,
                 test_instance,
                 transform: transforms.Compose=GRAPH_TRANSFORM,
                 hyper_params: dict = HYPER_PARAMS
                 ):
        
        # data directories
        self.data_root_dir = data_root_dir
        self.train_instance = train_instance
        self.val_instance = val_instance
        self.test_instance = test_instance
        self.transform = transform

        # result directories
        os.makedirs('graph', exist_ok=True)
        self.result_root_dir = os.path.join('graph', datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))
        os.makedirs(self.result_root_dir, exist_ok=True)
        
        # set up the hyperparameters
        self.LR = hyper_params['LR']
        self.BATCH_SIZE = hyper_params['BATCH_SIZE']
        self.EPOCHS = hyper_params['EPOCHS']

        # models
        self.curr_best_model = None
        self.curr_loss = float('inf')

        # results
        self.results = {"train_loss": [], "train_acc": [], "train_precision": [], "train_recall": [], "train_f1": [],
              "test_loss": [], "test_acc": [], "test_precision": [], "test_recall": [], "test_f1": []}
    
    def setup_device(self):
        # check if CUDA is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("[INFO] Using device: ", self.device)

    def setup_seed(self, seed=42):
        # set up the seed for reproducibility
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)

    def create_dataloaders(self, instance:str, shuffle=True):
        data = TimTremorGraph(self.data_root_dir, instance, self.transform)   
        class_names = data.classes

        NUM_WORKERS = os.cpu_count()

        data_loader = DataLoader(data,
                                batch_size=self.BATCH_SIZE,
                                shuffle=shuffle,
                                collate_fn=collate_fn,
                                num_workers=0,
                                pin_memory=False)

        return data_loader, class_names
    
    def setup_dataloaders(self):
        self.train_dataloader = []
        self.val_dataloader = []
        self.test_dataloader = []
        for instance in self.train_instance:
            train_dataloader, self.class_names = self.create_dataloaders(instance)
            self.train_dataloader.append(train_dataloader)

        for instance in self.val_instance:
            val_dataloader, _ = self.create_dataloaders(instance)
            self.val_dataloader.append(val_dataloader)

        for instance in self.test_instance:
            test_dataloader, _ = self.create_dataloaders(instance)
            self.test_dataloader.append(test_dataloader)

        self.num_classes = len(self.class_names)

    def setup_model(self):
        self.model = GraphProcessor(final_embd_dim=384).to(self.device)

    def setup_loss_function(self):
        self.loss_fn = nn.CrossEntropyLoss()

    def setup_optimizer(self):
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.LR)

    def train_step(self):
        self.model.train()

        train_loss = 0
        metrics = PerformanceMetric(self.num_classes)

        total_data_instance = 0

        for train_dataloader in self.train_dataloader:
            total_data_instance += len(train_dataloader)
            for batch, (X1, X2, y) in enumerate(train_dataloader):
                X1, X2, y = X1.to(self.device), X2.to(self.device), y.to(self.device)

                y_pred, _ = self.model(X1, X2)

                loss = self.loss_fn(y_pred, y)
                train_loss += loss.item()

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
                metrics.update(y_pred_class.cpu(), y.cpu())

        train_loss /= total_data_instance
        accuracy, precision, recall, f1, conf_matrix = metrics.compute()

        metrics.reset()

        return train_loss, accuracy, precision, recall, f1, conf_matrix
    
    def val_step(self):
        self.model.eval()

        val_loss = 0
        metrics = PerformanceMetric(self.num_classes)

        total_data_instance = 0

        with torch.inference_mode():
            for val_dataloader in self.val_dataloader:
                total_data_instance += len(val_dataloader)
                for batch, (X1, X2, y) in enumerate(val_dataloader):
                    X1, X2, y = X1.to(self.device), X2.to(self.device), y.to(self.device)

                    y_pred, _ = self.model(X1, X2)

                    loss = self.loss_fn(y_pred, y)
                    val_loss += loss.item()

                    y_pred_class = y_pred.argmax(dim=1)
                    metrics.update(y_pred_class.cpu(), y.cpu())

        val_loss /= total_data_instance
        accuracy, precision, recall, f1, conf_matrix = metrics.compute()

        metrics.reset()

        return val_loss, accuracy, precision, recall, f1, conf_matrix
    
    def test_step(self):
        self.curr_best_model.eval()

        test_loss = 0
        metrics = PerformanceMetric(self.num_classes)

        total_data_instance = 0

        with torch.inference_mode():
            for test_dataloader in self.test_dataloader:
                total_data_instance += len(test_dataloader)
                for batch, (X1, X2, y) in enumerate(test_dataloader):
                    X1, X2, y = X1.to(self.device), X2.to(self.device), y.to(self.device)

                    y_pred, _ = self.curr_best_model(X1, X2)

                    loss = self.loss_fn(y_pred, y)
                    test_loss += loss.item()

                    y_pred_class = y_pred.argmax(dim=1)
                    metrics.update(y_pred_class.cpu(), y.cpu())

        test_loss /= total_data_instance
        accuracy, precision, recall, f1, conf_matrix = metrics.compute()

        metrics.reset()

        return test_loss, accuracy, precision, recall, f1, conf_matrix

    def train(self):
        curr_results = {"train_loss": [], "train_acc": [], "train_precision": [], "train_recall": [], "train_f1": [],
              "test_loss": [], "test_acc": [], "test_precision": [], "test_recall": [], "test_f1": []}
        
        if self.curr_best_model is None:
            self.curr_best_model = self.model
            self.curr_loss = float('inf')

        for epoch in tqdm(range(self.EPOCHS)):
            train_loss, train_acc, train_precision, train_recall, train_f1, _ = self.train_step()
            test_loss, test_acc, test_precision, test_recall, test_f1, _ = self.val_step()

            curr_results["train_loss"].append(train_loss)
            curr_results["train_acc"].append(train_acc)
            curr_results["train_precision"].append(train_precision)
            curr_results["train_recall"].append(train_recall)
            curr_results["train_f1"].append(train_f1)

            curr_results["test_loss"].append(test_loss)
            curr_results["test_acc"].append(test_acc)
            curr_results["test_precision"].append(test_precision)
            curr_results["test_recall"].append(test_recall)
            curr_results["test_f1"].append(test_f1)

            if test_loss < self.curr_loss:
                self.curr_loss = test_loss
                self.curr_best_model = self.model

        return curr_results

    def save_train_results(self, results):
        self.results.update({k: self.results.get(k, []) + v for k, v in results.items()})
        
        # saving the results in a file - train_metrics.csv
        headers = list(self.results.keys())
        epoch_result = [list(col) for col in zip(*self.results.values())]

        csv_file_path = os.path.join(self.result_root_dir, 'train_metrics.csv')

        with open(csv_file_path, 'w', newline='') as csv_file:
            writer = csv.writer(csv_file)            
            writer.writerow(headers)            
            writer.writerows(epoch_result)

    def save_models(self):
        last = os.path.join(self.result_root_dir, 'last.pth')
        best = os.path.join(self.result_root_dir, 'best.pth')

        torch.save(obj=self.model.state_dict(), f=last)
        torch.save(obj=self.curr_best_model.state_dict(), f=best)

    def save_params(self):
        params = {
            'curr_loss': self.curr_loss
        }
        file_path = os.path.join(self.result_root_dir, 'params.txt')

        with open(file_path, 'w') as file:
            json.dump(params, file)        

    def run(self):
        # set up the device
        self.setup_device()

        # set up the seed
        self.setup_seed()

        # set up the data loaders
        self.setup_dataloaders()

        # set up the model
        self.setup_model()

        # set up the loss function
        self.setup_loss_function()

        # set up optimizer
        self.setup_optimizer()

        # train the model
        start_time = timer()
        curr_results = self.train()
        end_time = timer()
        print(f"[INFO] Training took {end_time - start_time:.3f} seconds")

        # save the results
        self.save_train_results(curr_results)

        # save the models
        self.save_models()

        # save hyperparameters
        self.save_params()

        # test the model
        test_loss, test_acc, test_precision, test_recall, test_f1, test_conf_mat = self.test_step()
        path = os.path.join(self.result_root_dir, 'test_metrics.txt')
        with open(path, 'w') as f:
            f.write(f"Test Loss: {test_loss}\n")
            f.write(f"Test Accuracy: {test_acc}\n")
            f.write(f"Test Precision: {test_precision}\n")
            f.write(f"Test Recall: {test_recall}\n")
            f.write(f"Test F1: {test_f1}\n")
            f.write(f"Test Confusion Matrix:\n{test_conf_mat.numpy()}\n")

    def resume(self, resume_dir):
        # set up the device
        self.setup_device()

        # set up the seed
        self.setup_seed()

        # set up the data loaders
        self.setup_dataloaders()

        # set up the model
        with open(os.path.join(resume_dir, 'params.txt'), 'r') as file:
            params = json.load(file)
            self.curr_loss = params['curr_loss']
        
        self.model = GraphProcessor()
        self.model.load_state_dict(torch.load(os.path.join(resume_dir, 'last.pth')))
        self.model = self.model.to(self.device)

        self.curr_best_model = GraphProcessor()
        self.curr_best_model.load_state_dict(torch.load(os.path.join(resume_dir, 'best.pth')))
        self.curr_best_model = self.curr_best_model.to(self.device)        

        self.results = pd.read_csv(os.path.join(resume_dir, 'train_metrics.csv')).to_dict(orient='list')

        # set up the loss function
        self.setup_loss_function()

        # set up optimizer
        self.setup_optimizer()

        # train the model
        start_time = timer()
        curr_results = self.train()
        end_time = timer()
        print(f"[INFO] Training took {end_time - start_time:.3f} seconds")

        # save the results
        self.save_train_results(curr_results)

        # save the models
        self.save_models()

        # save hyperparameters
        self.save_params()

        # test the model
        test_loss, test_acc, test_precision, test_recall, test_f1, test_conf_mat = self.test_step()
        path = os.path.join(self.result_root_dir, 'test_metrics.txt')
        with open(path, 'w') as f:
            f.write(f"Test Loss: {test_loss}\n")
            f.write(f"Test Accuracy: {test_acc}\n")
            f.write(f"Test Precision: {test_precision}\n")
            f.write(f"Test Recall: {test_recall}\n")
            f.write(f"Test F1: {test_f1}\n")
            f.write(f"Test Confusion Matrix:\n{test_conf_mat.numpy()}\n")
        

In [ ]:
data_root_dir = "Tim-Tremor\TimTremorMultimodal"
train_instance_path = ['graph_sensor_train.csv', 
                       'graph_sensor_train.csv', 
                       'graph_train.csv', 
                       'graph_train.csv']
val_instance_path = ['graph_sensor_val.csv', 
                       'graph_sensor_val.csv', 
                       'graph_val.csv', 
                       'graph_val.csv']
test_instance_path = ['graph_sensor_test.csv', 
                       'graph_sensor_test.csv', 
                       'graph_test.csv', 
                       'graph_test.csv']

trainer = Trainer(data_root_dir=data_root_dir,
                  train_instance=train_instance_path,
                  val_instance=val_instance_path,
                  test_instance=test_instance_path)
trainer.run()